In [2]:

import libs.config as config
model_env_name='MicroRTS'
train_net = config.create_net(model_env_name)    

In [ ]:
import torch 
import torch.nn as nn
from gym_microrts.envs.vec_env import MicroRTSVecEnv
from gym_microrts import microrts_ai
from collections import deque
import numpy as np
from torch.distributions.categorical import Categorical
import algo_envs.algo_base as AlgoBase
 
#模型及环境 MicroRTSEnv
MODEL_CONFIG = dict()
MODEL_CONFIG['env_name'] = "MicroRTSEnv" #其实用不到,只是为了区别
MODEL_CONFIG['num_envs'] = 8 # 环境数量 microRTS
MODEL_CONFIG['num_steps'] = 512 # 一次采样的长度
MODEL_CONFIG['obs_space'] = (10, 10, 27) # 状态空间 
MODEL_CONFIG['action_shape'] = [100, 6, 4, 4, 4, 4, 7, 49] # 动作空间
MODEL_CONFIG['device'] = torch.device('cuda:0' if torch.cuda.is_available() and False else 'cpu') # device
class CategoricalMasked(Categorical):
    def __init__(self, probs=None, logits=None, validate_args=None, masks=[], use_gpu = False):
        self.masks = masks
        self.device = torch.device('cuda' if torch.cuda.is_available() and use_gpu else 'cpu')
        if len(self.masks) == 0:
            super(CategoricalMasked, self).__init__(probs, logits, validate_args)
        else:
            self.masks = masks.type(torch.BoolTensor).to(self.device)
            logits = torch.where(self.masks, logits, torch.tensor(-1e+8).to(self.device))
            super(CategoricalMasked, self).__init__(probs, logits, validate_args)

    # H = sum(p(x)log(p(x)))
    def entropy(self):
        if len(self.masks) == 0:
            return super(CategoricalMasked, self).entropy()
        p_log_p = self.logits * self.probs
        p_log_p = torch.where(self.masks, p_log_p, torch.tensor(0.).to(self.device))
        return -p_log_p.sum(-1)
    
    def argmax(self):
        return torch.argmax(self.logits,dim=-1)


class MicroRTSAgent:
    def __init__(self,sample_net, is_checker=False):
        self.model_config = MODEL_CONFIG
        self.sample_net = sample_net
        self.num_envs = MODEL_CONFIG['num_envs']
        self.num_check_single_envs = 1
        self.num_steps = MODEL_CONFIG['num_steps']
        self.action_shape = MODEL_CONFIG['action_shape']
        self.outcomes = deque(maxlen=100)
        self.rewards = deque(maxlen=100)
        self.total_rewards = 0
        self.steps = 0
        if not is_checker:
            self.env = MicroRTSVecEnv(
                num_envs=self.num_envs,
                max_steps=5000,
                ai2s=[microrts_ai.coacAI for _ in range(self.num_envs)],
                map_path='maps/10x10/basesWorkers10x10.xml',
                reward_weight=np.array([10.0, 1.0, 1.0, 0.2, 1.0, 4.0])
            )
        else:
            self.env = self.env = MicroRTSVecEnv(
                num_envs=self.num_check_single_envs,
                max_steps=5000,
                ai2s=[microrts_ai.coacAI for _ in range(self.num_check_single_envs)],
                map_path='maps/10x10/basesWorkers10x10.xml',
                reward_weight=np.array([10.0, 1.0, 1.0, 0.2, 1.0, 4.0])
            )
        self.obs = self.env.reset()[0]
    
    def __del__(self):
        #del self.env
        pass
        
    # def get_units_number(unit_type, bef_obs, ind_obs):
    #     return int(bef_obs[ind_obs][:, :, unit_type].sum())
    
    @torch.no_grad()
    def _get_single_action(self,states, type_masks=None):
        
        split_logits, _ = self.sample_net(states)
        
        type_masks = torch.Tensor(type_masks)
        # print(logits.shape,type_masks.shape,len(split_logits),split_logits[0].shape)
        multi_categoricals = [CategoricalMasked(logits=split_logits[0], masks=type_masks)]
        action_components = [multi_categoricals[0].argmax()]
        
        # action_masks = torch.ones((20,78))
        action_masks = np.array(self.env.vec_client.getUnitActionMasks(action_components[0].cpu().numpy())).reshape(len(action_components[0]), -1)
        split_suam = torch.split(torch.Tensor(action_masks), self.action_shape[1:], dim=1)
        multi_categoricals = multi_categoricals + [CategoricalMasked(logits=logits, masks=iam) for (logits, iam) in
                                                    zip(split_logits[1:], split_suam)]
        action_components += [categorical.argmax() for categorical in multi_categoricals[1:]]
        action = torch.stack(action_components)
        
        return action.cpu().numpy()
        
    def sample_sa(self, Num=20000):

        states = []
        actions = []
        n = 0
        while n<Num:
            unit_mask = np.array(self.env.vec_client.getUnitLocationMasks()).reshape(1, -1)
            with torch.no_grad():
                action = self._get_single_action(states=torch.Tensor(self.obs), type_masks=unit_mask)
                next_obs, rs, done, truncated, _ = self.env.step(action.T)

                states.append(self.obs)
                actions.append(action.T)

            self.obs=next_obs
            n+=1
            
        return states, actions
    
agent= MicroRTSAgent(sample_net=train_net, is_checker=True)
states, actions = agent.sample_sa()

/opt/homebrew/lib/python3.9/site-packages/gym_microrts/microrts/maps/10x10/basesWorkers10x10.xml
[Lai.rewardfunction.RewardFunctionInterface;@517cd4b


In [13]:
len(states)

20000

In [14]:
states[0].shape, actions[0].shape

((1, 10, 10, 27), (1, 8))

In [9]:
len(agent.states),agent.states[0].shape

(51200, (16, 10, 10, 27))

In [ ]:
np.save('obs.npy',np.concatenate(agent.states,axis=0))
np.save('act.npy',np.concatenate(agent.actions,axis=0))